In [ ]:
# Load the libraries

import torch
from torch import nn        # Neural network building blocks (layers, activations, loss functions)
import numpy as np
import matplotlib.pyplot as plt
import time

# Show PyTorch version
torch.__version__

# Use current timestamp as the random seed for reproducibility across runs
SEED = int(time.time())

In [ ]:
# Setup device agnostic code
# Use GPU (CUDA) if available, otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{device} is in use")

## Binary classification

In [ ]:
# Import dataset generator and train/test splitter from scikit-learn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

In [ ]:
# Generate a 2D binary classification dataset shaped like two interleaving half-circles (moons)
# n_samples: total number of data points
# noise: standard deviation of Gaussian noise added to the data
# random_state: seed for reproducibility
X_dis, y_dis = make_moons(n_samples=1000, noise=0.02, random_state=SEED)
print(X_dis.shape)  # (1000, 2) — 1000 samples, 2 features (x, y coordinates)
print(y_dis.shape)  # (1000,)   — binary labels: 0 or 1

In [ ]:
# Visualize the moon dataset: each point is colored by its class label (0 or 1)
plt.scatter(X_dis[:,0], X_dis[:,1], s=1, c=y_dis, cmap='PiYG')
plt.show()

In [ ]:
# Convert NumPy arrays to PyTorch float tensors
# BCEWithLogitsLoss requires float targets for binary classification
X = torch.from_numpy(X_dis).type(torch.float)
y = torch.from_numpy(y_dis).type(torch.float)

type(X), X.dtype

In [ ]:
# Split dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=SEED)


In [ ]:
###  Model definition for binary classification on the moons dataset

class MoonsModule_v1(nn.Module):

    def __init__(self):
        super().__init__()

        # Four fully-connected layers: input(2) -> 25 -> 25 -> 25 -> output(1)
        # A single output neuron is standard for binary classification with BCEWithLogitsLoss
        self.layer_1 = nn.Linear(in_features=2, out_features=25)
        self.layer_2 = nn.Linear(in_features=25, out_features=25)
        self.layer_3 = nn.Linear(in_features=25, out_features=25)
        self.layer_4 = nn.Linear(in_features=25, out_features=1)
        self.relu    = nn.ReLU()   # Rectified Linear Unit: max(0, x)
        self.tahn    = nn.Tanh()   # Hyperbolic tangent: output in (-1, 1)


    def forward(self, x):
        # Forward pass: layer_1(ReLU) -> layer_2(Tanh) -> layer_3(ReLU) -> layer_4
        # Alternating activations help the network learn complex non-linear boundaries
        # Raw logits are returned; sigmoid is applied externally during inference
        return self.layer_4(  self.relu(  self.layer_3( self.tahn( self.layer_2( self.relu( self.layer_1(x) ) ) ) ) ) )

In [ ]:
# Instantiate the model and move it to the selected device (CPU or GPU)
model_1 = MoonsModule_v1().to(device)

In [ ]:
# model_1.state_dict()  # Uncomment to inspect initial layer weights and biases

In [ ]:
# Accuracy helper: computes percentage of correct predictions
def acc_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()  # Count matching elements
    acc = 100*correct/len(y_pred)                    # Convert to percentage
    return acc

In [ ]:
### Training loop for binary classification

# Set seeds for reproducibility on both CPU and GPU
torch.manual_seed( SEED )
torch.cuda.manual_seed( SEED )

epochs = 10

# BCEWithLogitsLoss combines Sigmoid + Binary Cross-Entropy in one numerically stable step
loss_fn = nn.BCEWithLogitsLoss()

# SGD (Stochastic Gradient Descent) optimizer with a moderate learning rate
optimizer = torch.optim.SGD(params=model_1.parameters(),
                            lr=0.05)

# Move data to the same device as the model
X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)


for epoch in range(epochs):

    # --- Training phase ---
    model_1.train()  # Enable dropout/batchnorm training behaviour (if any)

    y_logits = model_1(X_train).squeeze()          # Raw output; squeeze removes the trailing dim-1
    y_pred = torch.round(torch.sigmoid(y_logits))  # Sigmoid converts logits to [0,1]; round gives 0/1 labels

    loss = loss_fn(y_logits, y_train)  # Compute binary cross-entropy loss
    acc = acc_fn(y_train, y_pred)      # Training accuracy

    optimizer.zero_grad()  # Clear gradients from the previous step
    loss.backward()        # Backpropagate to compute gradients
    optimizer.step()       # Update weights

    # --- Evaluation phase ---
    model_1.eval()  # Disable dropout/batchnorm training behaviour

    with torch.inference_mode():  # Disable gradient tracking for efficiency

        test_logits = model_1(X_test).squeeze()
        test_pred = torch.round( torch.sigmoid(test_logits) )

        test_loss = loss_fn(test_logits, y_test)
        test_acc = acc_fn(y_test, test_pred)

        # Print metrics every 10 epochs (after the first two)
        if epoch > 1 and epoch%10 == 0:
            print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")   

In [ ]:
import requests
from pathlib import Path

# Download helper functions from the Learn PyTorch repo (only if not already present)
# plot_predictions: scatter plot of predictions vs ground truth
# plot_decision_boundary: meshgrid-based visualization of the model's decision regions
if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping downloading")
else:
  print("Downloading helper_functions.py")
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

from helper_functions import plot_predictions, plot_decision_boundary

In [ ]:
# Visualize how the model separates the two moon classes
# Left plot: decision boundary on the training set
# Right plot: decision boundary on the held-out test set
# A good model should show similar boundaries on both (no overfitting)
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_1, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_1, X_test, y_test)

### Multiclass classification

In [ ]:
# Spiral dataset from CS231n (Stanford) — a classic non-linearly-separable multiclass problem
# Three spiral arms, each belonging to one of K=3 classes
import numpy as np
N = 300  # Number of points per class
D = 2    # Input dimensionality (2D coordinates)
K = 3    # Number of classes
X = np.zeros((N*K, D))            # Feature matrix: (900, 2)
y = np.zeros(N*K, dtype='uint8')  # Integer class labels: 0, 1, or 2
for j in range(K):
  ix = range(N*j, N*(j+1))              # Index range for class j
  r = np.linspace(0.0, 1, N)            # Radius linearly increases from center outward
  t = np.linspace(j*4, (j+1)*4, N) + np.random.randn(N)*0.2  # Angle with small random noise
  X[ix] = np.c_[r*np.sin(t), r*np.cos(t)]  # Convert polar -> Cartesian coordinates
  y[ix] = j
# Visualize the spiral dataset; each color represents one class
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
plt.show()

In [ ]:
# Convert NumPy arrays to PyTorch tensors
# Features as float32; labels as LongTensor (required by CrossEntropyLoss)
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.LongTensor)

In [ ]:
# Split the spiral dataset: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=SEED)

In [ ]:
### Model definition for multiclass classification on the spiral dataset

class SpiralModule_v1(nn.Module):

    def __init__(self, input_features, hidden_layers, output_features):
        super().__init__()

        # nn.Sequential chains layers in order — cleaner than writing forward() manually
        # Architecture: Linear -> Tanh -> Linear -> ReLU -> Linear
        # output_features=3 gives one logit per class; argmax selects the predicted class
        self.linear_layers_stack = nn.Sequential(
            nn.Linear(in_features=input_features, out_features=hidden_layers),
            nn.Tanh(),   # Tanh works well in first layer for data centered around origin
            nn.Linear(in_features=hidden_layers, out_features=hidden_layers),
            nn.ReLU(),   # ReLU in deeper layers to prevent vanishing gradients
            nn.Linear(in_features=hidden_layers, out_features=output_features)
        )


    def forward(self, x):
        # Pass input through the sequential stack; output is raw class logits
        return self.linear_layers_stack(x)

In [ ]:
# Instantiate the spiral model: 2 inputs, 10 hidden units, 3 output classes
model_2 = SpiralModule_v1(input_features=2, 
                          hidden_layers=10, 
                          output_features=3).to(device)

In [ ]:
# model_2.state_dict()  # Uncomment to inspect initial layer weights and biases

In [ ]:
### Training loop for multiclass classification

# Set seeds for reproducibility
torch.manual_seed( SEED )
torch.cuda.manual_seed( SEED )

epochs = 100

# CrossEntropyLoss combines LogSoftmax + NLLLoss; expects raw logits and integer class labels
loss_fn = nn.CrossEntropyLoss()

# Adam optimizer: adaptive learning rates per parameter, often converges faster than SGD
optimizer = torch.optim.Adam(params=model_2.parameters(),
                            lr=0.1)

# Move data to the selected device
X_train, y_train = X_train.to(device), y_train.to(device)
X_test,  y_test  = X_test.to(device),  y_test.to(device)


for epoch in range(epochs+1):

    # --- Training phase ---
    model_2.train()

    y_logits = model_2(X_train)                           # Shape: (N, 3) — one logit per class
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1) # Softmax -> probabilities -> predicted class index

    loss = loss_fn(y_logits, y_train)  # Cross-entropy loss over all classes
    acc = acc_fn(y_train, y_pred)      # Training accuracy

    optimizer.zero_grad()  # Clear accumulated gradients
    loss.backward()        # Compute gradients via backpropagation
    optimizer.step()       # Update model parameters

    # --- Evaluation phase ---
    model_2.eval()

    with torch.inference_mode():  # No gradient tracking needed during evaluation

        test_logits = model_2(X_test).squeeze()
        test_pred =  torch.softmax(test_logits, dim=1).argmax(dim=1)

        test_loss = loss_fn(test_logits, y_test)
        test_acc = acc_fn(y_test, test_pred)

        # Log metrics every 10 epochs (after the first two)
        if epoch > 1 and epoch%10 == 0:
            print(f"Epoch: {epoch} | Loss: {loss:.5f}, Acc: {acc:.2f}% | Test Loss: {test_loss:.5f}, Test Acc: {test_acc:.2f}%")   

In [ ]:
# Visualize the spiral model's decision boundaries
# The model must learn curved, non-linear boundaries to separate the three spiral arms
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Train")
plot_decision_boundary(model_2, X_train, y_train)
plt.subplot(1, 2, 2)
plt.title("Test")
plot_decision_boundary(model_2, X_test, y_test)